In [ ]:
# config.py
import sys

# Configuration Parameters
NUM_DECKS = 2
DEFAULT_WAGER = 10
STARTING_BANKROLL = 100000
SHUFFLE_POINT = 0.25
TOTAL_CARDS = 52 * NUM_DECKS
NUM_SHOES_TO_PLAY = 1000
MAX_SPLITS = 3
# DAS
# INSURANCE
# SURRENDER
# RESPLIT ACES

# Verbose output flag
verbose = False

# Count values for cards
COUNT_VALUES = {
    '2': +1, '3': +1, '4': +1, '5': +1, '6': +1,
    '7': 0, '8': 0, '9': 0,
    '10': -1, 'J': -1, 'Q': -1, 'K': -1, 'A': -1
}

def log_message(msg):
    """Logs a message if verbose mode is enabled."""
    if verbose:
        print(msg)
        
RL_METHOD = "BasicStrategy"  #QLEARNING, Sarsa, BasicStrategy

In [ ]:
# environment.py
import random
import config

def create_deck():
    suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
    values = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
    single_deck = [(value, suit) for suit in suits for value in values]
    return single_deck * config.NUM_DECKS

def shuffle_deck(deck):
    random.shuffle(deck)
    return deck

def deal_card(deck, running_count):
    if len(deck) <= int(config.TOTAL_CARDS * config.SHUFFLE_POINT):
        config.log_message("Cut card reached! Reshuffle after this round.")

    card = deck.pop()
    card_val = config.COUNT_VALUES.get(card[0], 0)
    running_count += card_val

    # Compute true count
    decks_remaining = len(deck)/52.0
    if decks_remaining > 0:
        true_count = running_count/decks_remaining
    else:
        true_count = running_count
    true_count_int = int(round(true_count))

    return card, running_count, true_count_int

def deal_initial_hands(deck, running_count):
    # Deal to player: first card
    card, running_count, true_count_int = deal_card(deck, running_count)
    player_hand = [card]
    # Second card to player
    card, running_count, true_count_int = deal_card(deck, running_count)
    player_hand.append(card)
    # First card to dealer
    card, running_count, true_count_int = deal_card(deck, running_count)
    dealer_hand = [card]
    # Second card to dealer
    card, running_count, true_count_int = deal_card(deck, running_count)
    dealer_hand.append(card)

    # Return the hands, updated running_count, and the latest true_count_int
    return player_hand, dealer_hand, running_count, true_count_int

def reshuffle_if_needed(deck, running_count, shoes_played):
    if len(deck) <= int(config.TOTAL_CARDS * config.SHUFFLE_POINT):
        config.log_message("Reshuffling deck now...")
        deck[:] = shuffle_deck(create_deck())
        running_count = 0
        shoes_played += 1
        config.log_message(f"*** Completed Shoe {shoes_played} ***")
    return running_count, shoes_played

def calculate_hand_value(hand):
    value = 0
    aces_used_as_11 = 0
    card_values = {
        '2':2,'3':3,'4':4,'5':5,'6':6,'7':7,
        '8':8,'9':9,'10':10,'J':10,'Q':10,'K':10
    }

    for card, suit in hand:
        if card == 'A':
            value += 11
            aces_used_as_11 += 1
        else:
            value += card_values[card]

    while value > 21 and aces_used_as_11 > 0:
        value -= 10
        aces_used_as_11 -= 1

    is_soft = aces_used_as_11 > 0
    return value, is_soft

def is_blackjack(hand):
    value, _ = calculate_hand_value(hand)
    return len(hand) == 2 and value == 21

def can_split(hand):
    if len(hand) == 2:
        card_values = {
            '2': 2, '3': 3, '4':4, '5':5,'6':6,'7':7,
            '8':8,'9':9,'10':10,'J':10,'Q':10,'K':10,'A':11
        }
        return card_values[hand[0][0]] == card_values[hand[1][0]]
    return False

def dealer_turn(dealer_hand, deck, running_count):
    dealer_hits = 0
    while True:
        dealer_value, is_soft = calculate_hand_value(dealer_hand)
        config.log_message(f"Dealer's Hand: {dealer_hand} | Value: {dealer_value}")

        if dealer_value > 17:
            break
        elif dealer_value == 17 and is_soft:
            card, running_count, tc = deal_card(deck, running_count)
            dealer_hand.append(card)
            dealer_hits += 1
        elif dealer_value == 17:
            break
        else:
            card, running_count, tc = deal_card(deck, running_count)
            dealer_hand.append(card)
            dealer_hits += 1

    dealer_value, _ = calculate_hand_value(dealer_hand)
    return dealer_hand, dealer_value, running_count, dealer_hits

def is_pair(hand):
    return len(hand) == 2 and hand[0][0] == hand[1][0]

def get_card_value(card):
    card_rank = card[0]
    if card_rank in ['J','Q','K','10']:
        return 10
    elif card_rank == 'A':
        return 11
    else:
        return int(card_rank)

def dealer_upcard_value(card):
    if card[0] in ['J','Q','K','10']:
        return 10
    elif card[0] == 'A':
        return 11
    else:
        return int(card[0])

def determine_winner(player_value, dealer_value):
    config.log_message("\n--- Final Results ---")
    if player_value > 21:
        config.log_message("You busted. Dealer wins.")
    elif dealer_value > 21:
        config.log_message("Dealer busted. You win!")
    elif player_value > dealer_value:
        config.log_message("You win!")
    elif player_value < dealer_value:
        config.log_message("Dealer wins.")
    else:
        config.log_message("It's a push!")


In [ ]:
# strategy.py
import config
from environment import calculate_hand_value, is_pair, get_card_value, dealer_upcard_value, deal_card

def get_player_decision(hand, dealer_card, bankroll, current_wager, can_double, can_split_hand, splits_done, max_splits=config.MAX_SPLITS):
    player_value, is_soft = calculate_hand_value(hand)
    if player_value == 21:
        return 'stand'

    dealer_value = dealer_upcard_value(dealer_card)

    # Pair logic
    if is_pair(hand):
        rank = hand[0][0]
        pair_val = get_card_value(hand[0])
        if rank == 'A':
            if can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                if dealer_value in [5,6] and can_double and bankroll >= current_wager:
                    return 'double'
                return 'hit'

        if pair_val == 10:
            return 'stand'

        if pair_val == 9:
            if dealer_value in [2,3,4,5,6,8,9] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'stand'

        if pair_val == 8:
            if can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                if dealer_value in [2,3,4,5,6]:
                    return 'stand'
                else:
                    return 'hit'

        if pair_val == 7:
            if dealer_value in [2,3,4,5,6,7] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 6:
            if dealer_value in [2,3,4,5,6] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 5:
            if dealer_value in range(2,10) and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'

        if pair_val == 4:
            if dealer_value in [5,6] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 3:
            if dealer_value in range(2,8) and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 2:
            if dealer_value in range(2,8) and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

    # Soft totals
    if is_soft:
        if player_value == 20:
            return 'stand'
        if player_value == 19:
            if dealer_value == 6 and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'stand'
        if player_value == 18:
            if dealer_value in range(2,7) and can_double and bankroll >= current_wager:
                return 'double'
            elif dealer_value in [9,10,11]:
                return 'hit'
            else:
                return 'stand'
        if player_value == 17:
            if dealer_value in [3,4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 16:
            if dealer_value in [4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 15:
            if dealer_value in [4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 14:
            if dealer_value in [5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 13:
            if dealer_value in [5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'

    # Hard totals
    if not is_soft:
        if player_value >= 17:
            return 'stand'
        if player_value == 16:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 15:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 14:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 13:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 12:
            if dealer_value in [4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 11:
            if can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 10:
            if dealer_value in range(2,10) and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 9:
            if dealer_value in [3,4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value <= 8:
            return 'hit'

    return 'hit'

def player_action(deck, initial_hand, dealer_hand, bankroll, wager, running_count, max_splits=config.MAX_SPLITS):
    player_hands = [initial_hand]
    wagers = [wager]
    current_hand_index = 0
    splits_done = 0
    d_up = dealer_hand[0]

    while current_hand_index < len(player_hands):
        hand = player_hands[current_hand_index]

        # If hand has only one card, deal another
        if len(hand) == 1:
            card, running_count = deal_card(deck, running_count)
            hand.append(card)

        while True:
            hand_value, _ = calculate_hand_value(hand)
            if hand_value > 21:
                break

            can_double = (len(hand) == 2)
            can_split_hand = is_pair(hand)

            action = get_player_decision(hand, d_up, bankroll, wagers[current_hand_index], can_double, can_split_hand, splits_done, max_splits)

            if action == 'stand':
                break
            elif action == 'hit':
                card, running_count = deal_card(deck, running_count)
                hand.append(card)
            elif action == 'double':
                if can_double and bankroll >= wagers[current_hand_index]:
                    bankroll -= wagers[current_hand_index]
                    wagers[current_hand_index] *= 2
                    card, running_count = deal_card(deck, running_count)
                    hand.append(card)
                    break
                else:
                    # Couldn't double, just hit
                    card, running_count = deal_card(deck, running_count)
                    hand.append(card)
            elif action == 'split':
                if can_split_hand and splits_done < max_splits and bankroll >= wagers[current_hand_index]:
                    bankroll -= wagers[current_hand_index]
                    c1, c2 = hand[0], hand[1]
                    new_hand_1 = [c1]
                    new_hand_2 = [c2]
                    player_hands[current_hand_index] = new_hand_1
                    player_hands.insert(current_hand_index+1, new_hand_2)
                    wagers.append(wagers[current_hand_index])
                    splits_done += 1

                    # Deal new card to first split hand
                    card, running_count = deal_card(deck, running_count)
                    new_hand_1.append(card)
                    continue
                else:
                    # Couldn't split, just hit
                    card, running_count = deal_card(deck, running_count)
                    hand.append(card)

            hand_value, _ = calculate_hand_value(hand)
            if hand_value > 21:
                break
            if action in ['stand','double']:
                break

        current_hand_index += 1

    return player_hands, wagers, bankroll, running_count

def update_bankroll(bankroll, wager, outcome):
    if outcome == 'win':
        return bankroll + (wager * 2)
    elif outcome == 'push':
        return bankroll + wager
    elif outcome == 'lose':
        return bankroll
    else:
        raise ValueError("Invalid outcome provided.")

def initialize_bankroll(initial_amount=config.STARTING_BANKROLL):
    return initial_amount

def place_wager(bankroll, wager_amount=config.DEFAULT_WAGER):
    if wager_amount > bankroll:
        raise ValueError("Wager exceeds available bankroll.")
    return bankroll - wager_amount, wager_amount


In [ ]:
# rl_agent.py
import random
import config
import strategy

# Q-Learning Agent
class QLearningAgent:
    def __init__(self, actions, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.999):
        """
        actions: list of possible actions (e.g., ['hit', 'stand'])
        alpha: learning rate (0 < alpha <= 1)
        gamma: discount factor (0 <= gamma <= 1)
        epsilon: initial exploration rate (0 <= epsilon <= 1)
        epsilon_decay: factor by which epsilon is multiplied after each update 
                       to gradually reduce exploration.
                       For example, epsilon_decay=0.999 means epsilon * 0.999 each step.
        """
        self.actions = actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.q_table = {}  # Keys: (state, action), Value: Q-value (float)

    def get_q_value(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state, available_actions):
        """
        Epsilon-greedy action selection.
        With probability epsilon, choose a random available action.
        Otherwise, choose the action with the highest Q-value at this state.
        """
        if random.random() < self.epsilon:
            return random.choice(available_actions)
        else:
            q_values = [(self.get_q_value(state, a), a) for a in available_actions]
            _, best_action = max(q_values, key=lambda x: x[0])
            return best_action

    def update(self, state, action, reward, next_state):
        """
        Q-learning update:
        Q(s,a) = Q(s,a) + alpha * [r + gamma * max_a'(Q(s',a')) - Q(s,a)]
        """
        old_q = self.get_q_value(state, action)
        next_q_values = [self.get_q_value(next_state, a) for a in self.actions]
        best_next_q = max(next_q_values) if next_q_values else 0.0

        new_q = old_q + self.alpha * (reward + self.gamma * best_next_q - old_q)
        self.q_table[(state, action)] = new_q

        # Decay epsilon to reduce exploration over time
        self.epsilon = max(self.epsilon * self.epsilon_decay, 0.01)

# Sarsa Agent
class SarsaAgent:
    def __init__(self, actions, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.999):
        self.actions = actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.q_table = {}

    def get_q_value(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state, available_actions):
        if random.random() < self.epsilon:
            return random.choice(available_actions)
        else:
            q_values = [(self.get_q_value(state, a), a) for a in available_actions]
            _, best_action = max(q_values, key=lambda x: x[0])
            return best_action

    def update(self, state, action, reward, next_state, next_action):
        """
        Sarsa update rule:
        Q(s,a) = Q(s,a) + α [r + γ Q(s',a') - Q(s,a)]
        where a' is the action chosen in next_state.
        """
        old_q = self.get_q_value(state, action)
        next_q = self.get_q_value(next_state, next_action)
        new_q = old_q + self.alpha * (reward + self.gamma * next_q - old_q)
        self.q_table[(state, action)] = new_q

        # Decay epsilon
        self.epsilon = max(self.epsilon * self.epsilon_decay, 0.01)

# Basic Strategy Agent (no learning, just a policy)
class BasicStrategyAgent:
    def __init__(self, actions):
        self.actions = actions

    def choose_action(self, state, available_actions, player_hand, dealer_hand, bankroll, current_wager, splits_done=0):
        # Determine if double/split is possible
        can_double = (len(player_hand) == 2 and bankroll >= current_wager)
        can_split_hand = strategy.is_pair(player_hand)
        dealer_card = dealer_hand[0]

        action = strategy.get_player_decision(
            hand=player_hand,
            dealer_card=dealer_card,
            bankroll=bankroll,
            current_wager=current_wager,
            can_double=can_double,
            can_split_hand=can_split_hand,
            splits_done=splits_done,
            max_splits=config.MAX_SPLITS
        )

        # Ensure chosen action is in available_actions
        if action not in available_actions:
            if 'hit' in available_actions:
                action = 'hit'
            else:
                action = available_actions[0]

        return action



In [ ]:
# analytics.py
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import matplotlib.pyplot as plt
import statistics
import config
import io

class DataLogger:
    def __init__(self):
        self.records = []
        self.hand_counter = 0
        self.shoe_records = []
        self.epsilon_values = []  # To track epsilon over time

    def log_hand(self, outcome, profit, bankroll, actions_taken, starting_true_count, starting_decks_remaining, dealer_actions=[]):
        self.records.append({
            "hand_number": self.hand_counter,
            "outcome": outcome,
            "profit": profit,
            "bankroll": bankroll,
            "actions_taken": actions_taken,
            "dealer_actions": dealer_actions,
            "starting_true_count": starting_true_count,
            "starting_decks_remaining": starting_decks_remaining
        })
        self.hand_counter += 1

    def log_shoe_data(self, shoe_number, card_order):
        self.shoe_records.append({
            "shoe_number": shoe_number,
            "card_order": card_order
        })

    def log_epsilon(self, epsilon):
        # Call this method from main after each hand if agent has epsilon
        self.epsilon_values.append(epsilon)

    def get_data(self):
        return self.records[:]

    def get_outcomes(self):
        return [r["outcome"] for r in self.records]

    def get_profits(self):
        return [r["profit"] for r in self.records]

    def get_bankroll_history(self):
        return [r["bankroll"] for r in self.records]

    def get_counts(self):
        wins = sum(1 for r in self.records if r["outcome"] == "win")
        losses = sum(1 for r in self.records if r["outcome"] == "lose")
        pushes = sum(1 for r in self.records if r["outcome"] == "push")
        return wins, losses, pushes

def plot_bankroll_over_time(bankroll_history):
    plt.figure(figsize=(10,5))
    plt.plot(range(len(bankroll_history)), bankroll_history, label='Bankroll')
    plt.title("Bankroll Over Time")
    plt.xlabel("Hand Number")
    plt.ylabel("Bankroll")
    plt.grid(True)
    plt.legend()
    plt.savefig('static/bankroll_history.png')
    plt.close()

def plot_epsilon_convergence(logger: 'DataLogger'):
    # Plot epsilon over time if we have epsilon values
    if not logger.epsilon_values:
        return
    plt.figure(figsize=(10,5))
    plt.plot(range(len(logger.epsilon_values)), logger.epsilon_values, label='Epsilon')
    plt.title("Epsilon Convergence Over Time")
    plt.xlabel("Hand Number")
    plt.ylabel("Epsilon")
    plt.grid(True)
    plt.legend()
    plt.savefig('static/epsilon_convergence.png')
    plt.close()

def compute_ev_per_hand(profits):
    if not profits:
        return 0.0
    return sum(profits) / len(profits)

def compute_outcome_rates(wins, losses, pushes, total_hands):
    if total_hands == 0:
        return 0.0, 0.0, 0.0
    win_rate = wins / total_hands
    loss_rate = losses / total_hands
    push_rate = pushes / total_hands
    return win_rate, loss_rate, push_rate

def compute_variance(profits):
    if len(profits) <= 1:
        return 0.0
    return statistics.pvariance(profits)

def tc_to_bin(tc):
    if tc <= -7:
        return "≤ -7"
    elif tc >= 7:
        return "≥ 7"
    else:
        return str(tc)

def print_summary(logger: DataLogger):
    records = logger.get_data()
    total_hands = len(records)
    wins, losses, pushes = logger.get_counts()
    profits = [r["profit"] for r in records]
    ev = compute_ev_per_hand(profits)
    var = compute_variance(profits)
    win_rate, loss_rate, push_rate = compute_outcome_rates(wins, losses, pushes, total_hands)
    final_bankroll = records[-1]["bankroll"] if total_hands > 0 else config.STARTING_BANKROLL
    net_profit = final_bankroll - config.STARTING_BANKROLL
    wager = config.DEFAULT_WAGER
    ev_pct = (ev / wager) if wager != 0 else 0.0

    action_stats = {}
    for r in records:
        outcome = r["outcome"]
        for a in r.get("actions_taken", []):
            if a not in action_stats:
                action_stats[a] = {"win":0, "lose":0, "push":0, "total":0}
            action_stats[a][outcome] += 1
            action_stats[a]["total"] += 1

    for a, stats in action_stats.items():
        stats["win_rate"] = stats["win"] / stats["total"] if stats["total"] > 0 else 0
        stats["loss_rate"] = stats["lose"] / stats["total"] if stats["total"] > 0 else 0
        stats["push_rate"] = stats["push"] / stats["total"] if stats["total"] > 0 else 0

    tc_bins = {}
    for r in records:
        tc = r.get("starting_true_count", 0)
        bin_label = tc_to_bin(tc)
        outcome = r["outcome"]
        if bin_label not in tc_bins:
            tc_bins[bin_label] = {"win":0, "lose":0, "push":0, "total":0}
        tc_bins[bin_label][outcome] += 1
        tc_bins[bin_label]["total"] += 1

    for b, stats in tc_bins.items():
        stats["win_rate"] = stats["win"]/stats["total"] if stats["total"] > 0 else 0
        stats["loss_rate"] = stats["lose"]/stats["total"] if stats["total"] > 0 else 0
        stats["push_rate"] = stats["push"]/stats["total"] if stats["total"] > 0 else 0

    summary = {
        "total_hands": total_hands,
        "wins": wins,
        "win_rate": win_rate,
        "losses": losses,
        "loss_rate": loss_rate,
        "pushes": pushes,
        "push_rate": push_rate,
        "final_bankroll": final_bankroll,
        "net_profit": net_profit,
        "EV_per_hand": ev,
        "EV_percent": ev_pct,
        "variance": var,
        "risk_of_ruin": None,  # Placeholder: no calculation
        "action_stats": action_stats,
        "true_count_bins": tc_bins
    }

    return summary

def generate_strategy_chart_plot(agent):
    # Only 'hit' and 'stand' actions
    if hasattr(agent, 'set_greedy'):
        agent.set_greedy()

    dealer_upcards = [2,3,4,5,6,7,8,9,10,11]
    player_totals = list(range(5, 22))
    # Only hit and stand for chart
    ACTIONS = ['hit', 'stand']
    true_count_int = 0
    is_soft = 0

    action_map = {'hit':0, 'stand':1}
    data = []
    for p_total in player_totals:
        row = []
        for d_up in dealer_upcards:
            state = (p_total, d_up, is_soft, true_count_int)
            action = agent.choose_action(state, ACTIONS)
            row.append(action_map[action])
        data.append(row)

    plt.figure(figsize=(8, 6))
    plt.imshow(data, cmap='viridis', aspect='auto')
    plt.xticks(range(len(dealer_upcards)), dealer_upcards)
    plt.yticks(range(len(player_totals)), player_totals)
    plt.xlabel("Dealer Upcard")
    plt.ylabel("Player Total")
    plt.title("Strategy Chart (Hard Totals)")
    cbar = plt.colorbar()
    cbar.set_ticks([0,1])
    cbar.set_ticklabels(['hit','stand'])
    plt.savefig('static/strategy_chart.png')
    plt.close()


In [ ]:
# main.py
import config
import environment
import strategy
import analytics
from rl_agent import QLearningAgent, SarsaAgent, BasicStrategyAgent

def create_agent(actions):
    if config.RL_METHOD == "QLearning":
        return QLearningAgent(actions)
    elif config.RL_METHOD == "Sarsa":
        return SarsaAgent(actions)
    elif config.RL_METHOD == "BasicStrategy":
        return BasicStrategyAgent(actions)
    else:
        raise ValueError(f"Unknown RL method: {config.RL_METHOD}")

def main():
    logger = analytics.DataLogger()
    bankroll = strategy.initialize_bankroll()

    # RL agents: hit/stand only; BasicStrategy: hit/stand/double/split
    ACTIONS_RL = ['hit','stand']
    ACTIONS_BS = ['hit','stand','double','split']

    agent = create_agent(ACTIONS_RL if config.RL_METHOD != "BasicStrategy" else ACTIONS_BS)

    running_count = 0
    shoes_played = 0
    hands_played = 0
    splits_done = 0

    deck = environment.shuffle_deck(environment.create_deck())
    dealt_cards_this_shoe = []

    while shoes_played < config.NUM_SHOES_TO_PLAY and bankroll > 0:
        hand_starting_bankroll = bankroll
        try:
            bankroll, wager = strategy.place_wager(bankroll)
        except ValueError:
            break

        player_hand = []
        dealer_hand = []

        # Deal initial cards
        for _ in range(2):
            card, running_count, true_count_int = environment.deal_card(deck, running_count)
            player_hand.append(card)
            dealt_cards_this_shoe.append(card)

        for _ in range(2):
            card, running_count, true_count_int = environment.deal_card(deck, running_count)
            dealer_hand.append(card)
            dealt_cards_this_shoe.append(card)

        player_hands = [player_hand]
        wagers = [wager]
        current_hand_index = 0

        player_value, is_soft = environment.calculate_hand_value(player_hand)
        dealer_value, _ = environment.calculate_hand_value(dealer_hand)
        soft_flag = 1 if is_soft else 0
        starting_true_count = true_count_int
        starting_decks_remaining = len(deck)/52.0

        # Check dealer blackjack
        if environment.is_blackjack(dealer_hand):
            outcome = 'push' if environment.is_blackjack(player_hand) else 'lose'
            bankroll = strategy.update_bankroll(bankroll, wager, outcome)
            running_count, old_shoes_played = running_count, shoes_played
            running_count, shoes_played = environment.reshuffle_if_needed(deck, running_count, shoes_played)
            final_profit = bankroll - hand_starting_bankroll
            logger.log_hand(outcome, final_profit, bankroll, [], starting_true_count, starting_decks_remaining, dealer_actions=[])
            if shoes_played > old_shoes_played:
                logger.log_shoe_data(shoes_played, dealt_cards_this_shoe)
                dealt_cards_this_shoe = []
            hands_played += 1
            if hasattr(agent,'epsilon'):
                logger.log_epsilon(agent.epsilon)
            continue

        # Check player blackjack
        if environment.is_blackjack(player_hand):
            winnings = (wager * 1.5)
            bankroll += (winnings + wager)
            outcome = 'win'
            running_count, old_shoes_played = running_count, shoes_played
            running_count, shoes_played = environment.reshuffle_if_needed(deck, running_count, shoes_played)
            final_profit = bankroll - hand_starting_bankroll
            logger.log_hand(outcome, final_profit, bankroll, [], starting_true_count, starting_decks_remaining, dealer_actions=[])
            if shoes_played > old_shoes_played:
                logger.log_shoe_data(shoes_played, dealt_cards_this_shoe)
                dealt_cards_this_shoe = []
            hands_played += 1
            if hasattr(agent,'epsilon'):
                logger.log_epsilon(agent.epsilon)
            continue

        actions_this_hand = []
        dealer_actions = []
        last_state = None
        last_action = None

        while current_hand_index < len(player_hands):
            phand = player_hands[current_hand_index]
            while True:
                pval, is_soft = environment.calculate_hand_value(phand)
                if pval > 21:
                    break

                if config.RL_METHOD == "BasicStrategy":
                    available_actions = ['hit','stand']
                    can_double = (len(phand)==2 and bankroll >= wagers[current_hand_index])
                    can_split_hand = (environment.can_split(phand) and splits_done < config.MAX_SPLITS and bankroll >= wagers[current_hand_index])
                    if can_double:
                        available_actions.append('double')
                    if can_split_hand:
                        available_actions.append('split')
                else:
                    available_actions = ['hit','stand']

                state = (pval, dealer_value, 1 if is_soft else 0, true_count_int)
                if config.RL_METHOD == "BasicStrategy":
                    action = agent.choose_action(state, available_actions, phand, dealer_hand, bankroll, wagers[current_hand_index], splits_done)
                else:
                    # RL agents only know hit/stand
                    if 'double' in available_actions:
                        available_actions.remove('double')
                    if 'split' in available_actions:
                        available_actions.remove('split')
                    action = agent.choose_action(state, available_actions)

                actions_this_hand.append(action)
                last_state = state
                last_action = action

                if action == 'hit':
                    card, running_count, true_count_int = environment.deal_card(deck, running_count)
                    phand.append(card)
                    dealt_cards_this_shoe.append(card)
                    pval, is_soft = environment.calculate_hand_value(phand)
                    if pval > 21:
                        break

                elif action == 'stand':
                    break

                elif action == 'double' and config.RL_METHOD == "BasicStrategy":
                    if len(phand)==2 and bankroll>=wagers[current_hand_index]:
                        bankroll -= wagers[current_hand_index]
                        wagers[current_hand_index]*=2
                        card, running_count, true_count_int = environment.deal_card(deck, running_count)
                        phand.append(card)
                        dealt_cards_this_shoe.append(card)
                    break

                elif action == 'split' and config.RL_METHOD == "BasicStrategy":
                    if environment.can_split(phand) and splits_done<config.MAX_SPLITS and bankroll>=wagers[current_hand_index]:
                        bankroll -= wagers[current_hand_index]
                        c1,c2=phand[0], phand[1]
                        new_hand_1=[c1]
                        new_hand_2=[c2]
                        player_hands[current_hand_index]=new_hand_1
                        player_hands.insert(current_hand_index+1,new_hand_2)
                        wagers.insert(current_hand_index+1,wagers[current_hand_index])
                        splits_done+=1
                        card, running_count, true_count_int = environment.deal_card(deck, running_count)
                        new_hand_1.append(card)
                        dealt_cards_this_shoe.append(card)
                    else:
                        card, running_count, true_count_int = environment.deal_card(deck,running_count)
                        phand.append(card)
                        dealt_cards_this_shoe.append(card)
                        pval, is_soft = environment.calculate_hand_value(phand)
                        if pval>21:
                            break

            current_hand_index+=1

        dealer_must_play = any(environment.calculate_hand_value(h)[0]<=21 for h in player_hands)
        if dealer_must_play:
            dealer_hand_copy=dealer_hand.copy()
            dealer_hand_copy,dealer_value,running_count,dealer_hits=environment.dealer_turn(dealer_hand_copy,deck,running_count)
            if dealer_hits>0:
                dealer_actions=['hit']*dealer_hits
                if dealer_value<=21:
                    dealer_actions.append('stand')
            else:
                dealer_actions=['stand']
        else:
            dealer_actions=['stand']

        # Determine outcomes for each hand
        for i, phand in enumerate(player_hands):
            pval,_=environment.calculate_hand_value(phand)
            w=wagers[i]
            if pval<=21:
                if dealer_value>21 or pval>dealer_value:
                    outcome='win'
                    bankroll=strategy.update_bankroll(bankroll,w,'win')
                elif pval<dealer_value:
                    outcome='lose'
                    bankroll=strategy.update_bankroll(bankroll,w,'lose')
                else:
                    outcome='push'
                    bankroll=strategy.update_bankroll(bankroll,w,'push')
            else:
                outcome='lose'
                bankroll=strategy.update_bankroll(bankroll,w,'lose')

            final_profit=bankroll-hand_starting_bankroll

            # RL rewards and updates
            if config.RL_METHOD in ["QLearning","Sarsa"] and last_action is not None and last_state is not None:
                # Reward scheme: win=+1, push=0, lose=-1
                if outcome == 'win':
                    reward = 1.0
                elif outcome == 'push':
                    reward = 0.0
                else:
                    reward = -1.0

                next_state = (0,0,0,0) # Terminal
                if config.RL_METHOD == "QLearning":
                    # Normal Q-learning terminal update
                    agent.update(last_state, last_action, reward, next_state)
                else:
                    # Sarsa terminal update (no next_action)
                    old_q = agent.get_q_value(last_state, last_action)
                    new_q = old_q + agent.alpha * (reward - old_q)
                    agent.q_table[(last_state, last_action)] = new_q
                    # Manually decay epsilon for Sarsa after terminal
                    agent.epsilon = max(agent.epsilon * agent.epsilon_decay, 0.01)

            logger.log_hand(outcome,final_profit,bankroll,actions_this_hand,starting_true_count,starting_decks_remaining,dealer_actions)

        running_count, old_shoes_played=running_count,shoes_played
        running_count, shoes_played=environment.reshuffle_if_needed(deck,running_count,shoes_played)
        if shoes_played>old_shoes_played:
            logger.log_shoe_data(shoes_played,dealt_cards_this_shoe)
            dealt_cards_this_shoe=[]

        hands_played+=1
        if hasattr(agent,'epsilon') and config.RL_METHOD == "QLearning":
            # QLearning epsilon decay after each hand already happens in agent.update
            pass

    summary=analytics.print_summary(logger)
    bankroll_history=logger.get_bankroll_history()
    if bankroll_history:
        analytics.plot_bankroll_over_time(bankroll_history)

    if config.RL_METHOD in ["QLearning","Sarsa"]:
        analytics.plot_epsilon_convergence(logger)

    return {"hands_played":hands_played,"shoes_played":shoes_played,"summary":summary},agent,logger

if __name__=="__main__":
    main()


 

In [ ]:
# app.py

from flask import Flask, render_template, request, jsonify, Response, session, redirect, url_for
import config
from main import main as run_main_simulation
import analytics
import os

app = Flask(__name__)
app.secret_key = 'super_secret_key_for_session'

# Credentials (for simplicity)
ADMIN_USER = os.environ.get('ADMIN_USER', 'admin')
ADMIN_PASS = os.environ.get('ADMIN_PASS', 'password')

current_agent = None
current_logger = None
current_results = None

def login_required(f):
    def wrapper(*args, **kwargs):
        if not session.get('logged_in'):
            return Response("Unauthorized", 401)
        return f(*args, **kwargs)
    wrapper.__name__ = f.__name__
    return wrapper

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/login', methods=['GET','POST'])
def login():
    if request.method == 'POST':
        user = request.form.get('username')
        pwd = request.form.get('password')
        if user == ADMIN_USER and pwd == ADMIN_PASS:
            session['logged_in'] = True
            return redirect(url_for('index'))
        else:
            return "Invalid credentials", 401
    return render_template('login.html')

@app.route('/logout')
def logout():
    session.clear()
    return redirect(url_for('index'))

@app.route('/run_simulation', methods=['POST'])
@login_required
def run_simulation():
    rl_method = request.form.get('rl_method', 'BasicStrategy')
    num_decks = request.form.get('num_decks', '1')
    max_splits = request.form.get('max_splits', '3')

    num_decks = int(num_decks)
    max_splits = int(max_splits)

    config.RL_METHOD = rl_method
    config.NUM_DECKS = num_decks
    config.TOTAL_CARDS = 52 * config.NUM_DECKS
    config.MAX_SPLITS = max_splits

    results, agent, logger = run_main_simulation()
    global current_agent, current_logger, current_results
    current_agent = agent
    current_logger = logger
    current_results = results
    return jsonify({"hands_played": results["hands_played"], "shoes_played": results["shoes_played"]})

@app.route('/help')
def help_page():
    return render_template('help.html')

@app.route('/generate_report')
@login_required
def generate_report():
    global current_results, current_logger
    if current_results is None or current_logger is None:
        res, ag, lg = run_main_simulation()
        current_results = res
        current_logger = lg
    else:
        res = current_results
        lg = current_logger

    summary = res.get('summary', {})
    records = lg.get_data()  # Detailed hand-level records
    shoe_records = lg.shoe_records  # Shoe-level records

    import io
    output = io.StringIO()
    # Summary Section
    output.write("### Summary Metrics ###\n")
    output.write("Metric,Value\n")
    for key, value in summary.items():
        output.write(f"{key},{value}\n")

    output.write("\n### Hand-Level Records ###\n")
    # Include headers
    hand_headers = ["hand_number","outcome","profit","bankroll","actions_taken","dealer_actions","starting_true_count","starting_decks_remaining"]
    output.write(",".join(hand_headers)+"\n")
    for r in records:
        row = [
            str(r["hand_number"]),
            r["outcome"],
            str(r["profit"]),
            str(r["bankroll"]),
            "|".join(r["actions_taken"]) if r["actions_taken"] else "",
            "|".join(r["dealer_actions"]) if r["dealer_actions"] else "",
            str(r["starting_true_count"]),
            str(r["starting_decks_remaining"])
        ]
        output.write(",".join(row)+"\n")

    output.write("\n### Shoe-Level Records ###\n")
    # Include headers for shoe records
    output.write("shoe_number,card_order\n")
    for sr in shoe_records:
        card_order_str = "|".join([f"{c[0]}{c[1][0]}" for c in sr["card_order"]])  # e.g. "AHearts" -> "AH"
        output.write(f"{sr['shoe_number']},{card_order_str}\n")

    csv_data = output.getvalue()
    output.close()

    return Response(
        csv_data,
        mimetype="text/csv",
        headers={"Content-disposition": "attachment; filename=simulation_report.csv"}
    )

@app.route('/generate_strategy_chart_plot')
@login_required
def generate_strategy_chart_plot():
    global current_agent
    if current_agent is None:
        # If no agent yet, run simulation once
        r, a, l = run_main_simulation()
        current_agent = a

    analytics.generate_strategy_chart_plot(current_agent)
    return "Strategy chart generated!"

@app.route('/generate_epsilon_chart')
@login_required
def generate_epsilon_chart():
    if current_logger and current_logger.epsilon_values:
        analytics.plot_epsilon_convergence(current_logger)
        return "Epsilon chart generated!"
    else:
        return "No epsilon data available for current method!"

@app.route('/get_summary')
@login_required
def get_summary():
    if current_results:
        return jsonify(current_results["summary"])
    else:
        return jsonify({"error":"No results yet. Run simulation first."})

if __name__ == '__main__':
    app.run(debug=True)


In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8"/>
    <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
    <title>Blackjack Simulation UI</title>
    <style>
        body {
            font-family: Arial, sans-serif;
        }
        #side-panel {
            position: fixed;
            left: 0;
            top: 0;
            background: #f0f0f0;
            width: 200px;
            height: 100%;
            padding: 10px;
        }
        #main-content {
            margin-left: 220px;
            padding: 10px;
        }
        h1 {
            margin-top: 0;
        }
        .section {
            margin-bottom: 20px;
        }
    </style>
</head>
<body>
    <div id="side-panel">
        <h2>Menu</h2>
        <ul>
            <li><a href="/help" target="_blank">Help</a></li>
            <li><a href="/login" target="_blank">Login</a></li>
            <li><a href="/logout" target="_blank">Logout</a></li>
        </ul>
    </div>

    <div id="main-content">
        <h1>Blackjack Simulation</h1>
        <div class="section" id="parameters">
            <form id="params-form">
                <label for="rl_method">RL Method:</label>
                <select id="rl_method" name="rl_method">
                    <option value="QLearning">QLearning</option>
                    <option value="Sarsa">Sarsa</option>
                    <option value="BasicStrategy" selected>BasicStrategy</option>
                </select>
                <br><br>

                <label for="num_decks">Number of Decks (1-8):</label>
                <input type="number" id="num_decks" name="num_decks" min="1" max="8" value="1">
                <br><br>

                <label for="max_splits">Max Splits (max 3):</label>
                <input type="number" id="max_splits" name="max_splits" min="1" max="3" value="3">
            </form>
        </div>

        <div class="section" id="actions">
            <button type="button" onclick="runSimulation()">Run Simulation</button>
            <button type="button" onclick="generateReport()">Generate Report</button>
            <button type="button" onclick="generateStrategyChartPlot()">Generate Strategy Chart</button>
            <button type="button" onclick="generateEpsilonChart()">Show Epsilon Chart</button>
            <button type="button" onclick="toggleSummary()">Toggle Analytics Summary</button>
        </div>

        <div class="section" id="results">
            <div id="results">Run a simulation to see results...</div>
        </div>

        <div class="section" id="visuals">
            <img id="bankroll-plot" src="/static/bankroll_history.png" alt="Bankroll History" style="max-width:100%; height:auto;">
            <br><br>
            <img id="strategy-chart" src="" alt="Strategy Chart" style="max-width:100%; height:auto; display:none;">
            <br><br>
            <img id="epsilon-chart" src="" alt="Epsilon Chart" style="max-width:100%; height:auto; display:none;">
            <br><br>
            <pre id="summary-area" style="display:none;"></pre>
        </div>
    </div>

    <script>
    let summaryVisible = false;
    let summaryData = null;

    function runSimulation() {
        const form = document.getElementById('params-form');
        const formData = new FormData(form);

        fetch('/run_simulation', {
            method: 'POST',
            body: formData
        })
        .then(response => {
            if(response.status === 401) {
                alert("Please login first!");
                return;
            }
            return response.json();
        })
        .then(data => {
            if(!data) return;
            document.getElementById('results').textContent = `Simulation complete. Hands: ${data.hands_played}, Shoes: ${data.shoes_played}`;
            const plotImg = document.getElementById('bankroll-plot');
            plotImg.src = '/static/bankroll_history.png?' + new Date().getTime();
            document.getElementById('strategy-chart').style.display='none';
            document.getElementById('epsilon-chart').style.display='none';
            summaryVisible = false;
            summaryData = null;
            document.getElementById('summary-area').style.display='none';
            document.getElementById('summary-area').textContent='';
        });
    }

    function generateReport() {
        fetch('/generate_report')
        .then(response => {
            if(response.status === 401) {
                alert("Please login first!");
                return;
            }
            return response.blob();
        })
        .then(blob => {
            if(!blob) return;
            const url = window.URL.createObjectURL(blob);
            const a = document.createElement('a');
            a.style.display = 'none';
            a.href = url;
            a.download = 'simulation_report.csv';
            document.body.appendChild(a);
            a.click();
            window.URL.revokeObjectURL(url);
        });
    }

    function generateStrategyChartPlot() {
        fetch('/generate_strategy_chart_plot')
        .then(response => {
            if(response.status === 401) {
                alert("Please login first!");
                return;
            }
            return response.text();
        })
        .then(text => {
            if(!text) return;
            alert(text);
            const stratChart = document.getElementById('strategy-chart');
            stratChart.src = '/static/strategy_chart.png?' + new Date().getTime();
            stratChart.style.display = 'block';
        });
    }

    function generateEpsilonChart() {
        fetch('/generate_epsilon_chart')
        .then(response => {
            if(response.status === 401) {
                alert("Please login first!");
                return;
            }
            return response.text();
        })
        .then(text => {
            if(!text) return;
            alert(text);
            const epsChart = document.getElementById('epsilon-chart');
            epsChart.src = '/static/epsilon_convergence.png?' + new Date().getTime();
            epsChart.style.display = 'block';
        });
    }

    function toggleSummary() {
        if(summaryData === null) {
            fetch('/get_summary')
            .then(response => {
                if(response.status === 401) {
                    alert("Please login first!");
                    return;
                }
                return response.json();
            })
            .then(data => {
                if(!data || data.error) {
                    if(data && data.error) alert(data.error);
                    return;
                }
                summaryData = data;
                showOrHideSummary();
            });
        } else {
            showOrHideSummary();
        }
    }

    function showOrHideSummary() {
        summaryVisible = !summaryVisible;
        const area = document.getElementById('summary-area');
        if(summaryVisible) {
            area.textContent = JSON.stringify(summaryData, null, 2);
            area.style.display='block';
        } else {
            area.style.display='none';
        }
    }
    </script>
</body>
</html>




In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8"/>
    <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
    <title>Login</title>
</head>
<body>
    <h1>Login</h1>
    <form action="/login" method="POST">
        <label>Username:</label>
        <input type="text" name="username" required>
        <br><br>
        <label>Password:</label>
        <input type="password" name="password" required>
        <br><br>
        <button type="submit">Login</button>
    </form>
</body>
</html>


In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8"/>
    <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
    <title>Help & Documentation</title>
</head>
<body>
    <h1>Help & Documentation</h1>
    <p>Welcome to the Blackjack Simulation tool. Below is an overview of the available parameters and the features you can use to analyze and improve your blackjack strategies.</p>

    <h2>Parameters</h2>
    <ul>
        <li><strong>RL Method:</strong> Choose between QLearning, Sarsa, and BasicStrategy.
            <ul>
                <li><strong>QLearning</strong> and <strong>Sarsa</strong>: Reinforcement learning methods that learn optimal strategies over time. They start with exploration and gradually exploit learned policies.</li>
                <li><strong>BasicStrategy</strong>: A fixed, non-learning strategy derived from classical blackjack heuristics.</li>
            </ul>
        </li>
        <li><strong>Number of Decks (1-8):</strong> How many decks to use in the simulation. Increasing decks generally reduces count volatility.</li>
        <li><strong>Max Splits (1-3):</strong> The maximum number of times a pair can be split. Influences complexity and EV in advanced strategies.</li>
    </ul>

    <h2>Actions</h2>
    <ul>
        <li><strong>Run Simulation:</strong> After adjusting parameters, run a bulk simulation of blackjack hands. The system will simulate up to 1000 shoes (configurable) and record results.</li>
        <li><strong>Generate Report:</strong> Downloads a comprehensive CSV file including:
            <ul>
                <li>Summary metrics (win/loss rates, EV per hand, final bankroll).</li>
                <li>Hand-level records detailing each hand’s outcome, profit, actions taken, and dealer actions.</li>
                <li>Shoe-level data, showing the complete card order dealt in each shoe.</li>
            </ul>
        </li>
        <li><strong>Generate Strategy Chart:</strong> For RL methods, generates a heatmap-like chart (hit/stand decisions) reflecting the learned strategy for various player totals and dealer upcards.</li>
        <li><strong>Show Epsilon Chart:</strong> For QLearning or Sarsa, displays how the exploration rate (epsilon) changes over time, illustrating how the method transitions from exploring to exploiting the learned strategy.</li>
        <li><strong>Toggle Analytics Summary:</strong> Shows or hides an on-screen JSON summary of key performance metrics and distribution statistics, allowing quick reference without downloading a report.</li>
    </ul>

    <h2>Interpreting Results</h2>
    <p>Key metrics include:</p>
    <ul>
        <li><strong>win_rate / loss_rate / push_rate:</strong> Distribution of outcomes across the simulation.</li>
        <li><strong>EV_per_hand:</strong> Average profit or loss per hand, a crucial metric for long-term expectations.</li>
        <li><strong>final_bankroll / net_profit:</strong> Indicators of overall performance from the initial bankroll.</li>
        <li><strong>variance:</strong> A measure of result volatility. Higher variance means more swingy outcomes.</li>
    </ul>
    <p>You can experiment with different RL methods, deck counts, and splitting rules to see how strategies evolve and what outcomes are sustainable long-term. Use the CSV reports for detailed, offline analysis.</p>

    <p>For further guidance or troubleshooting, consult project documentation or reach out to the support team.</p>

    <a href="/">Back to Home</a>
</body>
</html>

